# 🚚 04. Compact Gurobi Vehicle Route Optimization

Zakaria’s MILP approach, optimized with two-index arcs rather than an arc copy for every vehicle. Every supplied site must be served. Volume, payload, shift, final unloading and depot return are enforced; no mid-route unloading or color-chamber constraints are claimed.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT / "Zakaria-Work"
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from Zakaria-Work or its parent directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from src.data_loader import load_and_clean_data, collection_summary
from src.clustering import extract_container_profiles, cluster_containers
from src.predictor import predict_container_demands, select_collection_sites
OUT = ROOT / "data" / "corrected"
OUT.mkdir(parents=True, exist_ok=True)
MAT_COLORS = {"white": "#1f77b4", "green": "#2ca02c", "brown": "#8c564b"}
plt.rcParams.update({"figure.figsize": (13, 5), "figure.dpi": 100})
from src.distance_provider import ModularDistanceProvider
from src.optimizer import solve_collection_routes
from src.visualizer import plot_interactive_routes


## 1. Select and explain candidates
All glass materials participate. Multiple selected bins at a site share travel but pay per-bin service. The explicit demonstration cap sorts by predicted fill and UUID, and exports every remaining selected site as deferred. Remove the cap with a suitable license/fleet to solve the complete batch.

In [ ]:
df = load_and_clean_data(ROOT / "glass_fill_levels_anonymized_f.csv")
print(f"{len(df):,} pre-emptying observations; {df.Location_id.nunique()} locations; "
      f"{len(df.groupby(['Location_id', 'material_type']))} observed bins")
print(f"Coverage: {df.date.min().date()} to {df.date.max().date()}")
display(df.head())

# Change this explicit morning dispatch date to replay a different day.
PLANNING_DATE = pd.Timestamp("2026-08-17")
profiles = extract_container_profiles(df, as_of=PLANNING_DATE, inclusive=False)
clustered_df, kmeans_model = cluster_containers(profiles, n_clusters=4)
print(f"Known bins strictly before dispatch: {len(profiles)}")

demands = predict_container_demands(clustered_df, planning_date=PLANNING_DATE,
    days_ahead=0, holiday_multiplier=1.0, container_volume_m3=3.0, glass_density_kg_m3=350.0)
display(demands[['Location_id','material_type','last_reading_date','last_pre_emptying_fill_perc',
                 'predicted_fill_perc','rate_source','urgency_status']].head(10))
print("Forecast at dispatch. Set days_ahead for an uncollected future scenario, not an observed future value.")

sites, decisions = select_collection_sites(demands)
DEMO_MAX_SITES = 24  # explicit demonstration subset, not a full-city result
urgent_candidates = sites.head(DEMO_MAX_SITES).copy()
deferred = sites.iloc[DEMO_MAX_SITES:].copy()
deferred['reason'] = 'deferred_by_explicit_demo_limit'
deferred.to_csv(OUT / 'deferred_sites.csv', index=False)
print(f'{len(sites)} selected sites: {len(urgent_candidates)} solver candidates, {len(deferred)} explicitly deferred.')


## 2. Stable geography and travel assumptions
A deterministic hash gives each synthetic site the same location in every subset. All distance legs and map markers share the same yard/disposal coordinates. Default drive time is the project’s ten-minute assumption; kilometer values remain illustrative haversine/detour distances, not road navigation.

In [ ]:
dist_provider = ModularDistanceProvider(urgent_candidates.Location_id.tolist(), fixed_travel_min=10)
print(dist_provider.mode, dist_provider.distance_provenance)


## 3. Fleet-first, distance-second MILP
Minimize used trucks first, then distance under that fleet. Shared volume-load and arrival variables replace vehicle-indexed assignment/order variables. With one uniform density, effective volume capacity is min(truck volume, payload/density), so both physical limits hold. MTZ prevents subtours even for zero-demand sites. Impossible arcs are pruned; returned loads and times are independently recomputed. A time-limited incumbent is labeled feasible, not proven optimal.

In [ ]:
result = solve_collection_routes(urgent_candidates, dist_provider,
    waste_density_kg_m3=350, truck_max_volume_m3=18, truck_max_weight_kg=12000,
    max_shift_hours=8, max_fleet_vehicles=12, service_time_min=5,
    unloading_time_min=15, time_limit_sec=20)
routes_df = result.routes
print(result.status, result.message)
print('Capacity lower bound:', result.fleet_lower_bound)
print('Model size:', result.model_stats)
display(routes_df)
display(result.unserved)
routes_df.to_csv(OUT / 'routes.csv', index=False)
result.unserved.to_csv(OUT / 'unserved_sites.csv', index=False)


## 4. Interactive proposal map
The Folium map labels synthetic assignments and direct lines. No maneuver instructions or real road distance are claimed.

In [ ]:
route_map = plot_interactive_routes(routes_df, urgent_candidates, dist_provider, OUT / 'route_proposal.html')
route_map


## 5. Operational result, not an invented savings estimate
Report actual proposal totals and every deferred/unserved site. No percentage savings or CO₂ reduction is inferred without a matched baseline and equivalent served demand.

In [ ]:
served = int(routes_df.num_stops.sum())
assert served + len(result.unserved) + len(deferred) == len(sites)
print(f'Sites: {served} served; {len(result.unserved)} solver-unserved; {len(deferred)} demo-deferred')
print(f'Trucks: {len(routes_df)}; modeled distance: {routes_df.total_distance_km.sum():.2f} km')
print(f'Working time: {routes_df.shift_hours.sum():.2f} h; collected mass: {routes_df.total_weight_kg.sum()/1000:.2f} t')
print('One load per truck, synthetic geography, assumed density and truck volume.')
